In [1]:
import os

# Move up one level to set the working directory to the repo root
os.chdir(os.path.abspath(os.path.join(os.getcwd(), "..")))

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

KAGGLE_DATA_PATH = Path(os.path.abspath(os.path.join(os.getcwd(), r"data\kaggle")))
SUBMISSION_DATA_PATH = Path(
    os.path.abspath(os.path.join(os.getcwd(), r"data\submissions"))
)
FIVETHIRTYEIGHT_DATA_PATH = Path(
    os.path.abspath(os.path.join(os.getcwd(), r"data\fivethirtyeight"))
)
BETEXPLORER_DATA_PATH = Path(
    os.path.abspath(os.path.join(os.getcwd(), r"data\betexplorer"))
)
DATA_PATH = Path(os.path.abspath(os.path.join(os.getcwd(), r"data")))

In [12]:
# Load data
submission_df = pd.read_csv(KAGGLE_DATA_PATH / "SampleSubmissionStage2.csv")
m_seeds = pd.read_csv(KAGGLE_DATA_PATH / "MNCAATourneySeeds.csv")
w_seeds = pd.read_csv(KAGGLE_DATA_PATH / "WNCAATourneySeeds.csv")
m_results = pd.read_csv(KAGGLE_DATA_PATH / "MNCAATourneyCompactResults.csv")
w_results = pd.read_csv(KAGGLE_DATA_PATH / "WNCAATourneyCompactResults.csv")
tourney_rounds = pd.read_csv(DATA_PATH / "tourney_round_lookup.csv")
# m_elo = pd.read_csv(FIVETHIRTYEIGHT_DATA_PATH  / "SBCB_BayesianElo_Men_20250319.csv")
# w_elo = pd.read_csv(FIVETHIRTYEIGHT_DATA_PATH  / "SBCB_BayesianElo_Women_20250319.csv")
m_elo = pd.read_csv(FIVETHIRTYEIGHT_DATA_PATH / "SBCB_PureElo_Men_20250319.csv")
w_elo = pd.read_csv(FIVETHIRTYEIGHT_DATA_PATH / "SBCB_PureElo_Women_20250319.csv")
m_odds = pd.read_csv(BETEXPLORER_DATA_PATH / "ncaam_betexplorer_odds_20250319.csv")
w_odds = pd.read_csv(BETEXPLORER_DATA_PATH / "ncaaw_betexplorer_odds_20250319.csv")
team_spellings = pd.read_csv(DATA_PATH / "TeamSpellings.csv", encoding="ISO-8859-1")


# Extract Season, Team1, and Team2 from ID
submission_df[["Season", "Team1", "Team2"]] = submission_df["ID"].str.split(
    "_", expand=True
)
submission_df[["Season", "Team1", "Team2"]] = submission_df[
    ["Season", "Team1", "Team2"]
].astype(int)


# Function to get seed values
def get_seed_values(seed):
    return int(seed[1:3])


m_seeds["SeedValue"] = m_seeds["Seed"].apply(get_seed_values)
w_seeds["SeedValue"] = w_seeds["Seed"].apply(get_seed_values)

mw_seeds = pd.concat([m_seeds, w_seeds], ignore_index=True)


# Merge Seeds
submission_df = submission_df.merge(
    mw_seeds, left_on=["Season", "Team1"], right_on=["Season", "TeamID"], how="left"
).rename(columns={"Seed": "Seed1", "SeedValue": "SeedValue1"})
submission_df = submission_df.merge(
    mw_seeds, left_on=["Season", "Team2"], right_on=["Season", "TeamID"], how="left"
).rename(columns={"Seed": "Seed2", "SeedValue": "SeedValue2"})
submission_df.drop(columns=["TeamID_x", "TeamID_y"], inplace=True)


# Ensure Seed1 and Seed2 are strings
submission_df["Seed1"] = submission_df["Seed1"].astype(str)
submission_df["Seed2"] = submission_df["Seed2"].astype(str)

# Create StrongSeed and WeakSeed
submission_df["StrongSeed"] = submission_df[["Seed1", "Seed2"]].min(axis=1)
submission_df["WeakSeed"] = submission_df[["Seed1", "Seed2"]].max(axis=1)


# Function to clean seed values
def clean_seed(seed):
    return seed[:-1] if seed[-1] in ["a", "b"] else seed


submission_df["StrongSeed"] = submission_df["StrongSeed"].apply(clean_seed)
submission_df["WeakSeed"] = submission_df["WeakSeed"].apply(clean_seed)

submission_df["StrongSeedValue"] = submission_df[["SeedValue1", "SeedValue2"]].min(
    axis=1
)
submission_df["WeakSeedValue"] = submission_df[["SeedValue1", "SeedValue2"]].max(axis=1)


# Merge Round Info
submission_df = submission_df.merge(
    tourney_rounds,
    left_on=["StrongSeed", "WeakSeed"],
    right_on=["StrongSeed", "WeakSeed"],
    how="left",
)

In [13]:
def process_results(results_df, seeds_df, rounds_df):
    results_df = results_df.merge(
        seeds_df,
        left_on=["Season", "WTeamID"],
        right_on=["Season", "TeamID"],
        how="left",
    ).rename(columns={"Seed": "WSeed", "SeedValue": "WSeedValue"})
    results_df = results_df.merge(
        seeds_df,
        left_on=["Season", "LTeamID"],
        right_on=["Season", "TeamID"],
        how="left",
    ).rename(columns={"Seed": "LSeed", "SeedValue": "LSeedValue"})
    results_df.drop(columns=["TeamID_x", "TeamID_y"], inplace=True)

    # Ensure Seed1 and Seed2 are strings
    results_df["WSeed"] = results_df["WSeed"].astype(str)
    results_df["LSeed"] = results_df["LSeed"].astype(str)

    # Create StrongSeed and WeakSeed
    results_df["StrongSeed"] = results_df[["WSeed", "LSeed"]].min(axis=1)
    results_df["WeakSeed"] = results_df[["WSeed", "LSeed"]].max(axis=1)

    results_df["StrongSeed"] = results_df["StrongSeed"].apply(clean_seed)
    results_df["WeakSeed"] = results_df["WeakSeed"].apply(clean_seed)

    results_df["StrongSeedValue"] = results_df[["WSeedValue", "LSeedValue"]].min(axis=1)
    results_df["WeakSeedValue"] = results_df[["WSeedValue", "LSeedValue"]].max(axis=1)

    # Identify if the StrongSeed won
    results_df["StrongSeedWon"] = (
        results_df["StrongSeedValue"] == results_df["WSeedValue"]
    )

    results_df = results_df.merge(rounds_df, on=["StrongSeed", "WeakSeed"], how="left")

    return results_df


m_results = process_results(m_results, m_seeds, tourney_rounds)
w_results = process_results(w_results, w_seeds, tourney_rounds)


def compute_seed_win_percentage(df):
    # Count total games for each seed matchup per round
    matchup_counts = (
        df.groupby(["StrongSeedValue", "WeakSeedValue", "Round"])
        .size()
        .reset_index(name="TotalGames")
    )

    # Count wins where the strong seed won
    win_counts = (
        df[df["StrongSeedWon"] == True]
        .groupby(["StrongSeedValue", "WeakSeedValue", "Round"])
        .size()
        .reset_index(name="Wins")
    )

    # Merge both counts to ensure all matchups exist
    merged = pd.merge(
        matchup_counts,
        win_counts,
        on=["StrongSeedValue", "WeakSeedValue", "Round"],
        how="left",
    ).fillna(0)

    # Convert Wins from float (due to fillna) to integer
    merged["Wins"] = merged["Wins"].astype(int)

    # Compute win percentage
    merged["StrongSeedWinPercentage"] = merged["Wins"] / merged["TotalGames"]

    return merged.sort_values(by=["Round", "StrongSeedValue", "WeakSeedValue"])


# Compute probabilities for men's and women's tournaments
m_seed_probs = compute_seed_win_percentage(m_results)
w_seed_probs = compute_seed_win_percentage(w_results)


# Merge Seed Probabilities
submission_df = submission_df.merge(
    m_seed_probs,
    left_on=["StrongSeedValue", "WeakSeedValue", "Round"],
    right_on=["StrongSeedValue", "WeakSeedValue", "Round"],
    how="left",
)


# Assign Seed Probability
submission_df["SeedProb"] = np.where(
    submission_df["SeedValue1"] == submission_df["StrongSeedValue"],
    submission_df["StrongSeedWinPercentage"],
    1 - submission_df["StrongSeedWinPercentage"],
)


# Function to get Elo-based probabilities
def elo_to_prob(elo1, elo2):
    return 1 / (1 + 10 ** ((elo2 - elo1) / 400))


m_elo["Team"] = m_elo["Team"].str.lower()
w_elo["Team"] = w_elo["Team"].str.lower()

# Merge Bayesian Elo Ratings
m_elo = m_elo.merge(
    team_spellings[["TeamNameSpelling", "MTeamID"]],
    left_on="Team",
    right_on="TeamNameSpelling",
    how="left",
).rename(columns={"MTeamID": "TeamID"})
w_elo = w_elo.merge(
    team_spellings[["TeamNameSpelling", "WTeamID"]],
    left_on="Team",
    right_on="TeamNameSpelling",
    how="left",
).rename(columns={"WTeamID": "TeamID"})

mw_elo = pd.concat([m_elo, w_elo], ignore_index=True)

submission_df = submission_df.merge(
    mw_elo[["TeamID", "Current Elo"]], left_on="Team1", right_on="TeamID", how="left"
).rename(columns={"Current Elo": "Elo1"})
submission_df = submission_df.merge(
    mw_elo[["TeamID", "Current Elo"]], left_on="Team2", right_on="TeamID", how="left"
).rename(columns={"Current Elo": "Elo2"})
submission_df["EloProb"] = submission_df.apply(
    lambda row: elo_to_prob(row["Elo1"], row["Elo2"]), axis=1
)
submission_df.drop(columns=["TeamID_x", "TeamID_y"], inplace=True)


m_odds["Team1"] = m_odds["Team1"].str.lower()
m_odds["Team2"] = m_odds["Team2"].str.lower()

# Merge Odds Data
m_odds = m_odds.merge(
    team_spellings[["TeamNameSpelling", "MTeamID"]],
    left_on="Team1",
    right_on="TeamNameSpelling",
    how="left",
).rename(columns={"MTeamID": "TeamID_1"})
m_odds = m_odds.merge(
    team_spellings[["TeamNameSpelling", "MTeamID"]],
    left_on="Team2",
    right_on="TeamNameSpelling",
    how="left",
).rename(columns={"MTeamID": "TeamID_2"})
m_odds.drop(columns=["TeamNameSpelling_x", "TeamNameSpelling_y"], inplace=True)

# Create TeamID_a (lower) and TeamID_b (higher)
m_odds["TeamID_a"] = m_odds[["TeamID_1", "TeamID_2"]].min(axis=1)
m_odds["TeamID_b"] = m_odds[["TeamID_1", "TeamID_2"]].max(axis=1)

# Assign Odds_a based on TeamID_a matching either TeamID_1 or TeamID_2
m_odds["Odds_a"] = np.where(
    m_odds["TeamID_a"] == m_odds["TeamID_1"], m_odds["Odds1"], m_odds["Odds2"]
)

# Assign Odds_b as the opposite of Odds_a
m_odds["Odds_b"] = np.where(
    m_odds["TeamID_b"] == m_odds["TeamID_1"], m_odds["Odds1"], m_odds["Odds2"]
)


w_odds["Team1"] = w_odds["Team1"].str.lower()
w_odds["Team2"] = w_odds["Team2"].str.lower()

w_odds["Team1"] = w_odds["Team1"].str.replace(r" w$", "", regex=True)
w_odds["Team2"] = w_odds["Team2"].str.replace(r" w$", "", regex=True)

# Merge Odds Data
w_odds = w_odds.merge(
    team_spellings[["TeamNameSpelling", "WTeamID"]],
    left_on="Team1",
    right_on="TeamNameSpelling",
    how="left",
).rename(columns={"WTeamID": "TeamID_1"})
w_odds = w_odds.merge(
    team_spellings[["TeamNameSpelling", "WTeamID"]],
    left_on="Team2",
    right_on="TeamNameSpelling",
    how="left",
).rename(columns={"WTeamID": "TeamID_2"})
w_odds.drop(columns=["TeamNameSpelling_x", "TeamNameSpelling_y"], inplace=True)

# Create TeamID_a (lower) and TeamID_b (higher)
w_odds["TeamID_a"] = w_odds[["TeamID_1", "TeamID_2"]].min(axis=1)
w_odds["TeamID_b"] = w_odds[["TeamID_1", "TeamID_2"]].max(axis=1)

# Assign Odds_a based on TeamID_a matching either TeamID_1 or TeamID_2
w_odds["Odds_a"] = np.where(
    w_odds["TeamID_a"] == w_odds["TeamID_1"], w_odds["Odds1"], w_odds["Odds2"]
)

# Assign Odds_b as the opposite of Odds_a
w_odds["Odds_b"] = np.where(
    w_odds["TeamID_b"] == w_odds["TeamID_1"], w_odds["Odds1"], w_odds["Odds2"]
)


mw_odds = pd.concat([m_odds, w_odds], ignore_index=True)


submission_df = submission_df.merge(
    mw_odds[["TeamID_a", "TeamID_b", "Odds_a", "Odds_b"]],
    left_on=["Team1", "Team2"],
    right_on=["TeamID_a", "TeamID_b"],
    how="left",
)

In [7]:
sbcb_bayesian_elo = submission_df.copy()

In [8]:
sbcb_bayesian_elo["Pred"] = sbcb_bayesian_elo["EloProb"]

In [11]:
sbcb_bayesian_elo["Pred"] = sbcb_bayesian_elo["Pred"].fillna(0.5)
sbcb_bayesian_elo[["ID", "Pred"]].to_csv(
    SUBMISSION_DATA_PATH / "sbcb_bayesian_elo.csv", index=False
)

In [14]:
sbcb_pure_elo = submission_df.copy()
sbcb_pure_elo["Pred"] = sbcb_pure_elo["EloProb"]
sbcb_pure_elo["Pred"] = sbcb_pure_elo["Pred"].fillna(0.5)
sbcb_pure_elo[["ID", "Pred"]].to_csv(
    SUBMISSION_DATA_PATH / "sbcb_pure_elo.csv", index=False
)

In [5]:
mw_elo.head()

,Unnamed: 0,Team,Conf.,Current Elo,Last,Season Min.,Season Max.,Home Court*,TeamNameSpelling,TeamID
0,1,duke,ACC,2118.670,+12 🟢@@11.91589355,1855.666,2118.670,80.61849,duke,1181.0
1,2,houston,Big 12,2116.650,+10 🟢@@9.635620117,1878.984,2116.650,84.03011,houston,1222.0
2,3,florida,SEC,2113.410,+16 🟢@@16.47436523,1807.053,2113.410,62.04281,florida,1196.0
3,4,auburn,SEC,2046.157,-19 🟠@@-18.93170166,1874.280,2115.515,67.18417,auburn,1120.0
4,5,alabama,SEC,2035.864,-38 🟠@@-38.12353516,1923.853,2102.856,99.86444,alabama,1104.0


In [6]:
submission_df[submission_df["Round"] == 1]

,ID,Pred,Season,Team1,Team2,Seed1,SeedValue1,Seed2,SeedValue2,StrongSeed,...,Wins,StrongSeedWinPercentage,SeedProb,Elo1,Elo2,EloProb,TeamID_a,TeamID_b,Odds_a,Odds_b
732,2025_1103_1112,0.5,2025,1103,1112,W13,13.0,W04,4.0,W04,...,123.0,0.788462,0.211538,1651.156,1952.972,0.149644,1103.0,1112.0,8.14,1.07
1322,2025_1104_1352,0.5,2025,1104,1352,W02,2.0,W15,15.0,W02,...,145.0,0.929487,0.929487,2035.864,1620.212,0.916268,1104.0,1352.0,1.01,18.11
1816,2025_1106_1120,0.5,2025,1106,1120,Y16a,16.0,Y01,1.0,Y01,...,154.0,0.987179,0.012821,1353.811,2046.157,0.018245,1106.0,1120.0,26.85,1.01
2941,2025_1110_1181,0.5,2025,1110,1181,W16a,16.0,W01,1.0,W01,...,154.0,0.987179,0.012821,1451.121,2118.670,0.020985,NaN,NaN,NaN,NaN
5109,2025_1116_1242,0.5,2025,1116,1242,Z10,10.0,Z07,7.0,Z07,...,95.0,0.612903,0.387097,1834.295,1907.924,0.395598,1116.0,1242.0,2.74,1.44
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
126882,2025_3380_3417,0.5,2025,3380,3417,Y16a,16.0,Y01,1.0,Y01,...,154.0,0.987179,0.012821,1425.191,2314.531,0.005944,NaN,NaN,NaN,NaN
128532,2025_3400_3456,0.5,2025,3400,3456,X01,1.0,X16b,16.0,X01,...,154.0,0.987179,0.987179,2322.997,1401.638,0.995052,NaN,NaN,NaN,NaN
129686,2025_3417_3471,0.5,2025,3417,3471,Y01,1.0,Y16b,16.0,Y01,...,154.0,0.987179,0.987179,2314.531,1589.023,0.984878,NaN,NaN,NaN,NaN
129924,2025_3422_3425,0.5,2025,3422,3425,Z16,16.0,Z01,1.0,Z01,...,154.0,0.987179,0.012821,1610.006,2313.850,0.017096,3422.0,3425.0,25.00,1.00


In [271]:
import goto_conversion


def apply_goto_conversion(df):
    """
    Applies the goto_conversion function to each row of a DataFrame with WOdds and LOdds.
    Handles missing or non-numeric values gracefully.
    Returns the DataFrame with added WProbability and LProbability columns.
    """

    def convert_row(row):
        try:
            # Convert WOdds and LOdds to floats (handles string values)
            Odds_a = float(row["Odds_a"]) if not pd.isnull(row["Odds_a"]) else np.nan
            Odds_b = float(row["Odds_b"]) if not pd.isnull(row["Odds_b"]) else np.nan

            # If odds are exactly 1.00, adjust them slightly to avoid errors
            if Odds_a == 1.00:
                Odds_a += 1e-6  # Tiny increment
            if Odds_b == 1.00:
                Odds_b += 1e-6  # Tiny increment

            # Check if either value is NaN (skip processing)
            if np.isnan(Odds_a) or np.isnan(Odds_b):
                return pd.Series({"Probability_a": np.nan, "Probability_b": np.nan})

            # Compute probabilities using goto_conversion
            listOfOdds = [Odds_a, Odds_b]
            probabilities = goto_conversion.goto_conversion(
                listOfOdds,
                total=1.0,
                multiplicativeIfUnprudentOdds=True,
                isAmericanOdds=False,
            )

            # Assign probabilities to the respective teams
            return pd.Series(
                {"Probability_a": probabilities[0], "Probability_b": probabilities[1]}
            )

        except Exception as e:
            # Handle unexpected errors gracefully
            print(f"Error processing row {row.name}: {e}")
            return pd.Series({"Probability_a": np.nan, "Probability_b": np.nan})

    # Convert entire columns to float (fix for entire dataset)
    df["Odds_a"] = pd.to_numeric(df["Odds_a"], errors="coerce")
    df["Odds_b"] = pd.to_numeric(df["Odds_b"], errors="coerce")

    # Apply conversion function to each row
    df[["Probability_a", "Probability_b"]] = df.apply(convert_row, axis=1)

    return df


# Convert odds to probabilities (apply_goto_conversion should be defined separately)
submission_df = apply_goto_conversion(submission_df)

In [309]:
submission_df[(submission_df["Round"] == 1) & (submission_df["StrongSeedValue"] == 1)]

,ID,Pred,Season,Team1,Team2,Seed1,SeedValue1,Seed2,SeedValue2,StrongSeed,...,Elo1,Elo2,EloProb,TeamID_a,TeamID_b,Odds_a,Odds_b,Probability_a,Probability_b,OddsProb
1816,2025_1106_1120,0.013905,2025,1106,1120,Y16a,16.0,Y01,1.0,Y01,...,1353.811,2046.157,0.018245,1106.0,1120.0,26.85,1.01,0.012419,0.987581,0.012419
2941,2025_1110_1181,0.014453,2025,1110,1181,W16a,16.0,W01,1.0,W01,...,1451.121,2118.670,0.020985,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6287,2025_1120_1384,0.986736,2025,1120,1384,Y01,1.0,Y16b,16.0,Y01,...,2046.157,1319.671,0.984962,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24267,2025_1181_1291,0.984676,2025,1181,1291,W01,1.0,W16b,16.0,W01,...,2118.670,1484.626,0.974664,1181.0,1291.0,1.01,51.00,0.989213,0.010787,0.989213
26194,2025_1188_1222,0.014610,2025,1188,1222,X16,16.0,X01,1.0,X01,...,1455.619,2116.650,0.021770,1188.0,1222.0,22.73,1.01,0.013050,0.986950,0.013050
28503,2025_1196_1313,0.983307,2025,1196,1313,Z01,1.0,Z16,16.0,Z01,...,2113.410,1522.152,0.967815,1196.0,1313.0,1.01,30.27,0.987973,0.012027,0.987973
99704,2025_3219_3400,0.012023,2025,3219,3400,X16a,16.0,X01,1.0,X01,...,1397.816,2322.997,0.004841,NaN,NaN,NaN,NaN,NaN,NaN,NaN
126478,2025_3376_3399,0.987071,2025,3376,3399,W01,1.0,W16,16.0,W01,...,2380.383,1640.085,0.986095,3376.0,3399.0,NaN,NaN,NaN,NaN,NaN
126882,2025_3380_3417,0.012133,2025,3380,3417,Y16a,16.0,Y01,1.0,Y01,...,1425.191,2314.531,0.005944,NaN,NaN,NaN,NaN,NaN,NaN,NaN
128532,2025_3400_3456,0.987967,2025,3400,3456,X01,1.0,X16b,16.0,X01,...,2322.997,1401.638,0.995052,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [273]:
# Assign Odds Probability
submission_df["OddsProb"] = np.where(
    submission_df["Team1"] == submission_df["TeamID_a"],
    submission_df["Probability_a"],
    submission_df["Probability_b"],
)

In [281]:
submission_df[["ID", "Pred", "SeedProb", "EloProb", "OddsProb"]].head()

,ID,Pred,SeedProb,EloProb,OddsProb
0,2025_1101_1102,0.5,NaN,0.761513,NaN
1,2025_1101_1103,0.5,NaN,0.258301,NaN
2,2025_1101_1104,0.5,NaN,0.036637,NaN
3,2025_1101_1105,0.5,NaN,0.903727,NaN
4,2025_1101_1106,0.5,NaN,0.658549,NaN


In [308]:
submission_df[(submission_df["Team1"] == 1101) & (submission_df["Team2"] == 1269)]

,ID,Pred,Season,Team1,Team2,Seed1,SeedValue1,Seed2,SeedValue2,StrongSeed,...,Elo1,Elo2,EloProb,TeamID_a,TeamID_b,Odds_a,Odds_b,Probability_a,Probability_b,OddsProb
159,2025_1101_1269,0.5,2025,1101,1269,nan,NaN,nan,NaN,nan,...,1467.915,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [293]:
# Final Probability Calculation using Weighted Sum Approach
def weighted_prob(row):
    weights = {"SeedProb": 0.3, "EloProb": 0.7, "OddsProb": 0.0}

    # Adjust weights for Round 1 high seeds
    if row["Round"] == 1 and row["Team1"] >= 3000 and int(row["StrongSeedValue"]) <= 3:
        weights = {"SeedProb": 0.6, "EloProb": 0.3, "OddsProb": 0.1}

    elif row["Round"] == 1 and row["Team1"] < 3000 and int(row["StrongSeedValue"]) <= 2:
        weights = {"SeedProb": 0.6, "EloProb": 0.3, "OddsProb": 0.1}

    elif row["Round"] == 1:
        weights = {"SeedProb": 0.2, "EloProb": 0.6, "OddsProb": 0.2}

    elif row["Round"] >= 2:
        weights = {"SeedProb": 0.1, "EloProb": 0.9, "OddsProb": 0.0}

    # Normalize weights based on available probabilities
    valid_probs = {
        key: row[key] for key in weights if key in row and not np.isnan(row[key])
    }
    if not valid_probs:
        return np.nan  # No available probabilities

    total_weight = sum(weights[key] for key in valid_probs)
    if total_weight == 0:
        return np.nan  # Avoid division by zero

    # Normalize weights and compute weighted probability
    normalized_weights = {key: weights[key] / total_weight for key in valid_probs}
    return sum(valid_probs[key] * normalized_weights[key] for key in valid_probs)

In [302]:
def weighted_prob(row):
    # Default weights
    weights = {"SeedProb": 0.3, "EloProb": 0.7, "OddsProb": 0.0}

    # Adjust for Round 1 top seeds
    if row["Round"] == 1 and int(row["StrongSeedValue"]) <= 2:
        weights = {"SeedProb": 0.8, "EloProb": 0.2, "OddsProb": 0.0}
    elif row["Round"] == 1 and int(row["StrongSeedValue"]) == 3:
        weights = {"SeedProb": 0.75, "EloProb": 0.25, "OddsProb": 0.0}

    # Adjust weights for later rounds
    elif row["Round"] == 2:
        weights = {"SeedProb": 0.25, "EloProb": 0.75, "OddsProb": 0.0}
    elif row["Round"] == 3:  # Sweet 16
        weights = {"SeedProb": 0.2, "EloProb": 0.8, "OddsProb": 0.0}
    elif row["Round"] == 4:  # Elite Eight
        weights = {"SeedProb": 0.15, "EloProb": 0.85, "OddsProb": 0.0}
    elif row["Round"] >= 5:  # Final Four & Championship
        weights = {"SeedProb": 0.05, "EloProb": 0.95, "OddsProb": 0.0}

    # Adjust for Women’s Tournament (if applicable)
    if row["Team1"] >= 3000:
        weights["SeedProb"] += 0.1
        weights["EloProb"] -= 0.1

    # Normalize weights based on available probabilities
    valid_probs = {
        key: row[key] for key in weights if key in row and not np.isnan(row[key])
    }
    if not valid_probs:
        return np.nan  # No available probabilities

    total_weight = sum(weights[key] for key in valid_probs)
    if total_weight == 0:
        return np.nan  # Avoid division by zero

    # Normalize weights and compute weighted probability
    normalized_weights = {key: weights[key] / total_weight for key in valid_probs}
    return sum(valid_probs[key] * normalized_weights[key] for key in valid_probs)

In [294]:
for i in range(5):  # Test first 5 rows
    print(f"Row {i}: {weighted_prob(submission_df.iloc[i])}")

Row 0: 0.7615129534981099
Row 1: 0.25830060560318224
Row 2: 0.03663680152537499
Row 3: 0.9037269494485601
Row 4: 0.6585491524027015


In [303]:
for i in (26194, 28503, 126478, 129924):  # Test first 5 rows
    print(f"Row {i}: {weighted_prob(submission_df.iloc[i])}")

Row 26194: 0.014610459767682268
Row 28503: 0.9833066781721256
Row 126478: 0.9870710297364773
Row 129924: 0.013248087049485446


In [304]:
submission_df["Pred"] = submission_df.apply(weighted_prob, axis=1)

In [307]:
# Ensure all NA values in Pred are set to 0.5
submission_df["Pred"] = submission_df["Pred"].fillna(0.5)

In [310]:
# Save final submission
submission_df[["ID", "Pred"]].to_csv(
    SUBMISSION_DATA_PATH / "Seed_Elo_Odds.csv", index=False
)